# Ismétlődő neurális hálózatok

Az előző modulban a szöveg gazdag szemantikai reprezentációiról volt szó. Az általunk használt architektúra megfogja a mondatban lévő szavak összegzett jelentését, de nem veszi figyelembe a szavak **sorrendjét**, mivel a beágyazásokat követő aggregációs művelet eltávolítja ezt az információt az eredeti szövegből. Mivel ezek a modellek nem képesek megjeleníteni a szórendet, nem tudnak megoldani bonyolultabb vagy kétértelmű feladatokat, például szöveggenerálást vagy kérdés-válasz feladatokat.

A szöveg szekvencia jelentésének megragadásához egy olyan neurális hálózati architektúrát fogunk használni, amelyet **ismétlődő neurális hálózatnak** (RNN) neveznek. RNN használatakor a mondatunkat egyesével adagoljuk át a hálózaton tokenenként, és a hálózat egy bizonyos **állapotot** ad ki, amit aztán a következő tokennel együtt újra bemenetként használunk.

![Kép, amely egy példát mutat be ismétlődő neurális hálózati generálásra.](../../../../../translated_images/hu/rnn.27f5c29c53d727b5.webp)

Adott egy $X_0,\dots,X_n$ token bemeneti sorozat, az RNN létrehoz egy neurális hálózati blokk-sorozatot, és ezt a sorozatot end-to-end módon tanítja visszaterjesztéssel. Minden hálózati blokk bemenetként párt vesz $(X_i,S_i)$, és eredményként előállítja $S_{i+1}$-et. A végső állapot $S_n$ vagy kimenet $Y_n$ egy lineáris osztályozóba kerül az eredmény előállításához. Minden hálózati blokk ugyanazokat a súlyokat használja, és egyetlen visszaterjesztő lépéssel tanítják end-to-end.

> A fenti ábra az ismétlődő neurális hálózatot kibontott formában (balra), és kompaktabb ismétlődő reprezentációban (jobbra) mutatja be. Fontos megérteni, hogy minden RNN cellának ugyanazok a **megosztható súlyai** vannak.

Mivel az állapotvektorok $S_0,\dots,S_n$ átvivődnek a hálózaton keresztül, az RNN képes megtanulni a szavak közötti sorrendi függőségeket. Például, ha a *not* szó szerepel valahol a szekvenciában, megtanulhat bizonyos elemeket az állapotvektorban tagadni.

Belül minden RNN cella két súlymátrixot tartalmaz: $W_H$ és $W_I$, valamint egy elfogultságot $b$. Minden RNN lépésnél az adott bemenet $X_i$ és bemeneti állapot $S_i$ alapján a kimeneti állapotot a következőképpen számítják ki: $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, ahol $f$ egy aktivációs függvény (gyakran $\tanh$).

> Olyan problémák esetén, mint a szöveggenerálás (amit a következő egységben tárgyalunk) vagy gépi fordítás, kimeneti értékre is szükség van minden RNN lépésnél. Ilyenkor van egy másik mátrix $W_O$, és a kimenet számítása így történik: $Y_i=f(W_O\times S_i+b_O)$.

Nézzük meg, hogyan segíthetnek az ismétlődő neurális hálózatok hír-adatállományunk osztályozásában.

> A sandbox környezetben futtatnunk kell a következő cellát, hogy megbizonyosodjunk arról, hogy a szükséges könyvtár telepítve van, és az adatok előre be vannak töltve. Ha helyben futtatja, ezt a cellát kihagyhatja.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Nagy modellek tanítása során a GPU memória allokáció problémát okozhat. Ezenkívül különböző minibatch méretekkel is kísérleteznünk kell, hogy az adatok beleférjenek a GPU memóriánkba, miközben a tanítás elegendően gyors marad. Ha ezt a kódot saját GPU gépeden futtatod, kísérletezhetsz a minibatch méret állításával a tanítás felgyorsítása érdekében.

> **Megjegyzés**: Bizonyos NVidia driverváltozatok ismerten nem szabadítják fel a memóriát a modell tanítása után. Több példát is futtatunk ebben a jegyzetfüzetben, és ez bizonyos beállítások mellett memória kimerüléshez vezethet, különösen, ha te magad futtatsz kísérleteket ugyanabban a jegyzetfüzetben. Ha furcsa hibákat tapasztalsz a modell tanításának megkezdésekor, érdemes lehet újraindítani a jegyzetfüzet kerneljét.


In [3]:
batch_size = 16
embed_size = 64

## Egyszerű RNN osztályozó

Egy egyszerű RNN esetében minden rekurzív egység egy egyszerű lineáris hálózat, amely bemenetként egy vektort és egy állapotvektort fogad, és egy új állapotvektort állít elő. Keras-ban ezt az `SimpleRNN` réteggel lehet ábrázolni.

Bár közvetlenül is átadhatunk one-hot kódolt tokeneket az RNN rétegnek, ez nem jó ötlet a magas dimenzionalitásuk miatt. Ezért egy beágyazási réteget használunk a szóvektorok dimenziójának csökkentésére, amit egy RNN réteg követ, majd végül egy `Dense` osztályozó.

> **Megjegyzés**: Ha a dimenzionalitás nincs annyira magas, például karakter-alapú tokenizáció esetén, lehet értelme, hogy közvetlenül one-hot kódolt tokeneket adjunk az RNN cellának.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Megjegyzés:** Egyszerűség kedvéért itt egy nem tanított beágyazási réteget használunk, de jobb eredmények érdekében használhatunk előre betanított beágyazási réteget a Word2Vec segítségével, ahogy azt az előző egységben leírtuk. Jó gyakorlat lenne, ha átírnád ezt a kódot úgy, hogy előre betanított beágyazásokkal működjön.

Most tanítsuk meg az RNN-ünket. Az RNN-ek általában elég nehezen taníthatók, mert miután az RNN cellákat kibontjuk a sorozathossz mentén, a visszaterjesztésben érintett rétegek száma meglehetősen nagy lesz. Ezért kisebb tanulási rátát kell választanunk, és a hálózatot nagyobb adathalmazon kell tanítani, hogy jó eredményeket érjünk el. Ez elég sokáig tarthat, ezért javasolt GPU használata.

A gyorsítás érdekében az RNN modellt csak a hírcímeken fogjuk tanítani, kihagyva a leírást. Kipróbálhatod a leírással való tanítást is, és megláthatod, tudod-e betanítani a modellt.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Megjegyzés:** A pontosság valószínűleg itt alacsonyabb lesz, mert csak hírcímeken tanulunk.


## Változó hosszúságú szekvenciák újbóli áttekintése 

Ne felejtsük el, hogy az `TextVectorization` réteg automatikusan kitölti a minibatch-ben szereplő változó hosszúságú szekvenciákat kitöltő tokenekkel. Kiderült, hogy ezek a tokenek is részt vesznek a tanításban, és megnehezíthetik a modell konvergenciáját.

Több megközelítést is alkalmazhatunk a kitöltés mennyiségének minimalizálására. Az egyik az, hogy a dataset-et szekvenciahossz szerint rendezzük és méret alapján csoportosítjuk az összes szekvenciát. Ezt az `tf.data.experimental.bucket_by_sequence_length` függvénnyel lehet elvégezni (lásd a [dokumentációt](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)).

Egy másik megközelítés a **maszkolás** használata. A Kerasban egyes rétegek támogatják egy extra bemenet beillesztését, amely megmutatja, hogy mely tokeneket vegyük figyelembe a tanítás során. A maszkolás beépítéséhez a modellbe vagy külön `Masking` réteget tehetünk be ([dokumentáció](https://keras.io/api/layers/core_layers/masking/)), vagy megadhatjuk az `mask_zero=True` paramétert az `Embedding` rétegünk számára.

> **Megjegyzés**: Ez a tanítás az egész adathalmazon körülbelül 5 percet vesz igénybe egy epochra. Bármikor megszakíthatod a tanítást, ha elfogy a türelmed. Az is megoldás lehet, hogy korlátozod az oktatásra használt adatok mennyiségét azzal, hogy az `.take(...)` feltételt hozzáadod az `ds_train` és `ds_test` adatállományokhoz.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Mivel most maszkolást használunk, a modellt az összes cím és leírás adatkészleten tudjuk tanítani.

> **Megjegyzés**: Észrevetted, hogy egy olyan vektorizálót használtunk, amit csak a hírek címei alapján tanítottak, nem az egész cikk tartalma alapján? Ez potenciálisan azt eredményezheti, hogy néhány token figyelmen kívül marad, ezért jobb lenne a vektorizálót újratanítani. Azonban ennek valószínűleg csak nagyon kis hatása lehet, így az egyszerűség kedvéért az előzőleg előtanított vektorizálónál maradunk.


## LSTM: Hosszú távú rövidtávú memória

Az RNN-ek egyik fő problémája a **gradiens eltűnés**. Az RNN-ek lehetnek egészen hosszúak, és nehezen tudják visszavezetni a grádienseket a hálózat első rétegéhez a visszafelé terjesztés során. Amikor ez megtörténik, a hálózat nem tud kapcsolatokat tanulni távoli tokenek között. Ennek a problémának az elkerülésére az egyik megoldás az **explicit állapotkezelés** bevezetése a **kapuk** használatával. A két leggyakoribb architektúra, amely kapukat vezet be, a **hosszú távú rövidtávú memória** (LSTM) és a **gated relay unit** (GRU). Itt az LSTM-ekkel foglalkozunk.

![Kép, amely egy példát mutat egy hosszú távú rövidtávú memória cellára](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Egy LSTM hálózat hasonlóan van szervezve, mint egy RNN, de két állapotot továbbítanak rétegről rétegre: a tényleges állapotot, $c$-t, és a rejtett vektort, $h$-t. Minden egységnél a rejtett vektor $h_{t-1}$ összevonódik a bemenettel $x_t$, és együtt szabályozzák, hogy mi történik az állapottal $c_t$ és a kimenettel $h_{t}$ a **kapukon** keresztül. Minden kapu szigmoid aktivációval rendelkezik (kimenet a $[0,1]$ tartományban), amely bitenkénti maszkként tekinthető, amikor megszorozzuk az állapotvektorral. Az LSTM-ek a következő kapukkal rendelkeznek (balról jobbra a fenti képen):
* **felejtő kapu**, amely meghatározza, hogy az állapotvektor $c_{t-1}$ mely összetevőit kell elfelejtenünk, és melyeket továbbengednünk.
* **bemeneti kapu**, amely meghatározza, hogy a bemeneti vektorból és az előző rejtett vektorból mennyi információt kell beépíteni az állapotvektorba.
* **kimeneti kapu**, amely a új állapotvektort veszi és eldönti, hogy annak mely összetevőit használjuk a új rejtett vektor $h_t$ előállításához.

Az állapot $c$ összetevőit úgy képzelhetjük el, mint zászlókat, amelyeket be- és kikapcsolhatunk. Például amikor a *Alice* nevet látjuk a szekvenciában, feltételezzük, hogy nőre utal, és felhúzzuk azt a zászlót az állapotban, amely azt jelzi, hogy nőnemű főnév van a mondatban. Amikor pedig találunk további szavakat, például *and Tom*, akkor felhúzzuk azt a zászlót, amely a többes számú főnévre utal. Így az állapot manipulálásával követni tudjuk a mondat grammatikai tulajdonságait.

> **Megjegyzés**: Ez egy nagyszerű forrás az LSTM-ek belső működésének megértéséhez: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) Christopher Olah-tól.

Bár egy LSTM cella belső szerkezete bonyolultnak tűnhet, a Keras ezt az implementációt elrejti az `LSTM` rétegben, így a fenti példában nekünk csak annyi a dolgunk, hogy lecseréljük a rekurzív réteget:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Megjegyzés:** Az LSTM-ek kiképzése is meglehetősen lassú, és a képzés elején nem tűnhet úgy, hogy jelentős pontosságnövekedés történik. Jó pontosság eléréséhez némi ideig folytatni kell a kiképzést.


## Kétirányú és többrétegű RNN-ek

Eddigi példáinkban a visszatérő hálózatok egy szekvencia kezdetétől a végéig működnek. Ez számunkra természetesnek tűnik, mert ugyanabban az irányban halad, mint ahogy olvasunk vagy hallgatjuk a beszédet. Azonban olyan helyzetekben, ahol véletlenszerű hozzáférés szükséges a bemeneti szekvenciához, értelmesebb a visszatérő számítást mindkét irányban futtatni. Azok az RNN-ek, amelyek engedik a számításokat mindkét irányban, **kétirányú** RNN-eknek nevezik őket, és úgy hozhatók létre, hogy a visszatérő réteget egy speciális `Bidirectonal` réteggel csomagoljuk be.

> **Megjegyzés**: Az `Bidirectional` réteg két másolatot készít a benne lévő rétegről, és az egyik példány `go_backwards` tulajdonságát `True`-ra állítja, ezzel a szekvencia mentén ellentétes irányba futtatja.

A visszatérő hálózatok, legyenek akár egyirányúak vagy kétirányúak, mintázatokat ragadnak meg a szekvencián belül, és ezeket állapotvektorokban tárolják vagy kimenetként adják vissza. Ahogy a konvolúciós hálózatoknál, itt is építhetünk további visszatérő réteget az első után, hogy magasabb szintű mintázatokat ragadjunk meg, amelyek alacsonyabb szintű mintázatokból épülnek fel, melyeket az első réteg kivont. Ez vezet minket a **többrétegű RNN** fogalmához, amely két vagy több visszatérő hálózatból áll, ahol az előző réteg kimenete kerül bemenetként a következő réteghez.

![Kép egy többrétegű hosszú-rövid távú memóriás RNN-ről](../../../../../translated_images/hu/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Kép Fernando López [e csodálatos bejegyzéséből](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3).*

A Keras megkönnyíti ezen hálózatok felépítését, mert csak több visszatérő réteget kell hozzáadnunk a modellhez. Az utolsó réteg kivételével minden rétegnél meg kell adnunk az `return_sequences=True` paramétert, mert szükségünk van arra, hogy a réteg visszaadjon minden köztes állapotot, nem csak a visszatérő számítás végső állapotát.

Építsünk egy két rétegű kétirányú LSTM-et az osztályozási problémánkhoz.

> **Megjegyzés** ez a kód ismét meglehetősen sok időt vesz igénybe a befejezéshez, de a legjobb pontosságot adja, amit eddig láttunk. Szóval talán megéri várni és megnézni az eredményt.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN-ek más feladatokra

Eddig az RNN-eket elsősorban szövegszekvenciák osztályozására használtuk. De sok más feladatot is képesek ellátni, mint például szöveg generálása és gépi fordítás — ezeket a feladatokat a következő egységben fogjuk tárgyalni.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
